In [268]:
import inspect
from collections import namedtuple

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [269]:

# ── Synthetic waveform generator ───────────────────────────────────────────────
def _adapt_rpm_profile_fn(profile):
    """Normalize profile to callable(angle_deg, elapsed_ms) -> float RPM."""
    if not callable(profile):
        rpm_value = float(profile)
        return lambda angle_deg, elapsed_ms: rpm_value

    try:
        sig = inspect.signature(profile)
        params = list(sig.parameters.values())
        has_varargs = any(p.kind == inspect.Parameter.VAR_POSITIONAL for p in params)
        positional_count = sum(
            p.kind in (inspect.Parameter.POSITIONAL_ONLY, inspect.Parameter.POSITIONAL_OR_KEYWORD)
            for p in params
        )
        accepts_time = has_varargs or positional_count >= 2
    except (TypeError, ValueError):
        accepts_time = False

    if accepts_time:
        return lambda angle_deg, elapsed_ms: float(profile(angle_deg, elapsed_ms))

    return lambda angle_deg, elapsed_ms: float(profile(angle_deg))


def gen_synthetic_waveform(rpm, n_revs, wheel, noise_ms=0.0):
    """
    Generate synthetic rising-edge timestamps for a trigger wheel.

    Parameters
    ----------
    rpm      : float or callable(cumulative_angle_deg[, elapsed_time_ms]) -> float
               Constant RPM (float) or a function returning instantaneous RPM
               as a function of cumulative crank angle [degrees], optionally
               with elapsed time [ms] as a second argument.
               Use the rpm_* helpers below to build common profiles.
    n_revs   : int            — number of full revolutions to simulate
    wheel    : WheelDescriptor — geometry of the trigger wheel
    noise_ms : float          — std-dev of Gaussian timing jitter added per edge [ms]

    Returns
    -------
    times     : pd.Series  — rising-edge timestamps [ms]
    intervals : pd.Series  — per-tooth durations     [ms]
    """
    rpm_fn = _adapt_rpm_profile_fn(rpm)
    tooth_angles = np.asarray(wheel.tooth_angles_deg, dtype=float)
    n_teeth = len(tooth_angles)
    n_steps = int(n_revs) * n_teeth

    times = np.empty(n_steps, dtype=float)
    intervals = np.empty(n_steps, dtype=float)

    idx = 0
    t_ms = 0.0
    angle_deg = 0.0

    for i in range(n_steps):
        times[i] = t_ms

        tooth_angle = tooth_angles[idx]
        rpm_now = rpm_fn(angle_deg, t_ms)
        if rpm_now <= 0.0:
            raise ValueError(
                f"RPM profile produced non-positive RPM={rpm_now:.6f} at angle={angle_deg:.3f}°, t={t_ms:.3f} ms"
            )

        dt_ms = tooth_angle * 60_000.0 / (360.0 * rpm_now)
        if noise_ms > 0.0:
            dt_ms = max(dt_ms + np.random.normal(0.0, noise_ms), 1e-6)

        intervals[i] = dt_ms
        t_ms += dt_ms
        angle_deg += tooth_angle
        idx = (idx + 1) % n_teeth

    return pd.Series(times), pd.Series(intervals)


def gen_synthetic_signal(times, intervals, duty=0.4):
    """
    Build a plottable square-wave signal from rising-edge timestamps and intervals.

    Parameters
    ----------
    times     : rising-edge timestamps [ms]
    intervals : per-tooth durations [ms]  (from gen_synthetic_waveform)
    duty      : fraction of each interval spent high (default 0.4)

    Returns
    -------
    (signal_times, signal_values) as pd.Series pairs
    """
    t = np.asarray(times, dtype=float)
    dt = np.asarray(intervals, dtype=float)
    n = len(t)

    sig_t = np.empty(3 * n, dtype=float)
    sig_v = np.empty(3 * n, dtype=float)

    sig_t[0::3] = t
    sig_t[1::3] = t
    sig_t[2::3] = t + dt * duty

    sig_v[0::3] = 0.0
    sig_v[1::3] = 1.0
    sig_v[2::3] = 0.0

    return pd.Series(sig_t), pd.Series(sig_v)


def inject_signal_glitches(signal_times, signal_values, n_glitches,
                           glitch_width, glitch_min_offset=None, seed=None):
    """
    Inject short spurious pulses into a (time, signal) waveform.

    Each glitch is a brief HIGH pulse inserted into a LOW region of the signal,
    simulating a noise spike that crosses the detection threshold.  Because the
    glitch is inserted into the raw waveform, all downstream code — edge
    detection, period computation, decoder — sees it naturally.

    Parameters
    ----------
    signal_times    : array-like — waveform timestamps (any consistent unit)
    signal_values   : array-like — waveform values (0 / 1)
    n_glitches      : int        — number of spurious pulses to inject
    glitch_width    : float      — duration of each glitch pulse (same unit as signal_times)
    glitch_min_offset : float or None
                       Minimum gap between the start of a low window and the
                       glitch placement.  Defaults to glitch_width.
    seed            : int or None — RNG seed for reproducibility

    Returns
    -------
    (signal_times, signal_values) — pd.Series pair with glitches merged in,
    sorted by time.  Drop-in replacement for the original arrays.

    Notes
    -----
    Glitches are placed only in genuinely LOW regions that are at least
    2 * glitch_width wide, so the pulse fits without overlapping the
    flanking real edges.
    """
    rng = np.random.default_rng(seed)
    if glitch_min_offset is None:
        glitch_min_offset = glitch_width

    t = np.asarray(signal_times, dtype=float)
    v = np.asarray(signal_values, dtype=float)

    # Find low samples that have enough room for a glitch pulse on both sides
    low_mask = v < 0.5
    low_t = t[low_mask]

    # Keep only positions where a full glitch fits before the next rising edge
    # Conservatively require offset + width < distance to next point
    valid_mask = low_t <= (low_t[-1] - glitch_width - glitch_min_offset) if len(low_t) > 1 else np.array([], dtype=bool)
    valid_t = low_t[valid_mask]

    if len(valid_t) == 0:
        return pd.Series(t), pd.Series(v)

    # Pick random host positions and offset each by glitch_min_offset + random extra
    host_t = valid_t[rng.integers(0, len(valid_t), size=n_glitches)]
    glitch_starts = host_t + glitch_min_offset

    # Two points per glitch: rising (0→1) then falling (1→0)
    glitch_t = np.empty(2 * n_glitches, dtype=float)
    glitch_v = np.empty(2 * n_glitches, dtype=float)
    glitch_t[0::2] = glitch_starts
    glitch_t[1::2] = glitch_starts + glitch_width
    glitch_v[0::2] = 1.0
    glitch_v[1::2] = 0.0

    merged_t = np.concatenate([t, glitch_t])
    merged_v = np.concatenate([v, glitch_v])
    sort_idx = np.argsort(merged_t, kind='stable')
    return pd.Series(merged_t[sort_idx]), pd.Series(merged_v[sort_idx])


def drop_signal_teeth(signal_times, signal_values, n_drops, seed=None):
    """
    Simulate lost teeth by removing rising-edge pulses from a (time, signal) waveform.

    Each dropped tooth removes one HIGH pulse (rising edge + falling edge) and
    merges the resulting gap into the surrounding LOW region, so the next real
    tooth's measured interval spans two tooth periods instead of one.

    Parameters
    ----------
    signal_times  : array-like — waveform timestamps (any consistent unit)
    signal_values : array-like — waveform values (0 / 1)
    n_drops       : int        — number of teeth (HIGH pulses) to remove
    seed          : int or None — RNG seed for reproducibility

    Returns
    -------
    (signal_times, signal_values) — pd.Series pair with teeth removed.
    Drop-in replacement for the original arrays.

    Notes
    -----
    The first and last tooth pulses are never dropped so that the waveform
    remains well-bounded.  If n_drops exceeds the number of available interior
    pulses, as many as possible are dropped without replacement.
    """
    rng = np.random.default_rng(seed)

    t = np.asarray(signal_times, dtype=float)
    v = np.asarray(signal_values, dtype=float)

    # Find rising-edge indices: 0→1 transitions
    rising_idx = np.flatnonzero((v[1:] > 0.5) & (v[:-1] <= 0.5)) + 1

    # Exclude the very first and very last pulse to keep the waveform bounded
    interior_rising = rising_idx[1:-1]

    if len(interior_rising) == 0:
        return pd.Series(t), pd.Series(v)

    n_actual = min(n_drops, len(interior_rising))
    drop_rising = rng.choice(interior_rising, size=n_actual, replace=False)

    # For each chosen rising edge, find and mark both the rising and the
    # immediately following falling edge (first 1→0 transition after it)
    drop_mask = np.zeros(len(t), dtype=bool)
    for r in drop_rising:
        # Mark the rising sample itself
        drop_mask[r] = True
        # Walk forward to find the matching falling edge
        for j in range(r + 1, len(v)):
            if v[j] <= 0.5:
                drop_mask[j] = True
                break
            # Mark any intermediate HIGH samples (e.g. the constant-1 plateau)
            drop_mask[j] = True

    keep_mask = ~drop_mask
    return pd.Series(t[keep_mask]), pd.Series(v[keep_mask])


# ── RPM profile helpers ────────────────────────────────────────────────────────
def rpm_const(rpm):
    """Constant-speed profile."""
    return lambda angle: rpm


def rpm_linear(rpm_start, accel_rpm_per_rev):
    """
    Linearly-accelerating profile.
    accel_rpm_per_rev: RPM change per full revolution (positive = accelerating).
    """
    return lambda angle: rpm_start + accel_rpm_per_rev * (angle / 360.0)


def rpm_linear_ms(rpm_start, accel_rpm_per_ms):
    """Linearly-accelerating profile in elapsed time (ms)."""
    return lambda angle, t_ms: rpm_start + accel_rpm_per_ms * t_ms


def rpm_sine(rpm_base, amplitude, period_deg=360.0, phase_deg=0.0):
    """
    Sinusoidally-modulated RPM profile.
    rpm_base   : mean RPM
    amplitude  : peak RPM deviation (must be < rpm_base to keep RPM positive)
    period_deg : period of the sine modulation in crank-angle degrees
                 (default 360 = one oscillation per revolution)
    phase_deg  : phase offset [degrees]
    """
    phase_rad = np.deg2rad(phase_deg)
    return lambda angle: rpm_base + amplitude * np.sin(2 * np.pi * angle / period_deg + phase_rad)


def rpm_staged_ms(stages, fallback=None):
    """
    Compose a staged RPM profile over elapsed time.

    Parameters
    ----------
    stages : iterable of (duration_ms, profile_fn)
             profile_fn can be either callable(angle) or callable(angle, stage_time_ms).
             duration_ms can be None/np.inf for an open-ended stage.
    fallback : callable(angle[, stage_time_ms]) or None
               Used after all finite stages. If None, reuses the final stage profile.

    Returns
    -------
    callable(angle, time_ms) -> rpm
    """
    normalized = []
    t_start = 0.0

    for duration_ms, profile_fn in stages:
        duration = np.inf if duration_ms is None else float(duration_ms)
        if duration <= 0.0 and not np.isinf(duration):
            raise ValueError(f"Stage duration must be > 0 ms (or None/inf), got {duration}")

        t_end = np.inf if np.isinf(duration) else (t_start + duration)
        normalized.append((t_start, t_end, _adapt_rpm_profile_fn(profile_fn)))

        if np.isinf(duration):
            break
        t_start = t_end

    if not normalized:
        raise ValueError("At least one stage is required")

    fallback_fn = _adapt_rpm_profile_fn(fallback if fallback is not None else stages[-1][1])

    def profile(angle, time_ms):
        t = float(time_ms)
        for t0, t1, fn in normalized:
            if t < t1:
                return fn(angle, t - t0)

        last_end = normalized[-1][1]
        fallback_t = t - last_end if np.isfinite(last_end) else 0.0
        return fallback_fn(angle, fallback_t)

    return profile


In [321]:

# ── Synthetic data parameters ──────────────────────────────────────────────────
SYN_RPM          = 300.0   # base RPM
SYN_N_REVS       = 100      # number of full revolutions to simulate
SYN_TOOTH_OFFSET = 0       # tooth index assigned to the first edge by the filter

# ── Wheel geometry ─────────────────────────────────────────────────────────────
# Explicit crank angle [deg] from each real tooth to the next.
# Edit individual values here to model non-uniform tooth spacing.
# Must sum to 360°.  32 entries — one per real tooth.
# Pattern: 11 teeth | 20° gap | 11 teeth | 20° gap | 10 teeth | 30° sync gap
TOOTH_ANGLES_DEG = np.array([
    13.5, 10., 10., 10., 10., 10., 10., 10., 10., 10.,    # teeth  0-9
    16.5,                                                 # tooth 10  (2-slot gap -> 20°)
    13.5, 10., 10., 10., 10., 10., 10., 10., 10., 10.,    # teeth 11-20
    16.5,                                                 # tooth 21  (2-slot gap -> 20°)
    13.5, 10., 10., 10., 10., 10., 10., 10., 10.,         # teeth 22-30
    26.5,                                                 # tooth 31  (3-slot sync gap -> 30°)
])
assert len(TOOTH_ANGLES_DEG) == 32, f"Expected 32 angles, got {len(TOOTH_ANGLES_DEG)}"
assert abs(TOOTH_ANGLES_DEG.sum() - 360.0) < 1e-6, f"Angles sum to {TOOTH_ANGLES_DEG.sum():.4f}°, expected 360°"
print("Tooth angles [deg]:", TOOTH_ANGLES_DEG)
print(f"Sum = {TOOTH_ANGLES_DEG.sum():.1f}°")


# ── Wheel descriptor ───────────────────────────────────────────────────────────
# Mirrors TriggerWaveform fields used for sync detection:
#   tooth_angles_deg    ≡  per-tooth angle increments
#   sync_ratio_from/to  ≡  synchronizationRatioFrom[] / synchronizationRatioTo[]
#                          one entry per consecutive gap tracked (gapTrackingLength)
#   sync_gap_deg        ≡  angle of the sync-gap tooth, used to bootstrap the tracker
WheelDescriptor = namedtuple('WheelDescriptor', [
    'tooth_angles_deg',  # np.ndarray: crank angle [deg] from each tooth to the next
    'sync_ratio_from',   # list[float]: lower bounds of gap-ratio window(s) for sync detection
    'sync_ratio_to',     # list[float]: upper bounds of gap-ratio window(s)
    'sync_gap_deg',      # float: angle spanned by the sync-gap tooth [deg]
])

WHEEL_6G75 = WheelDescriptor(
    tooth_angles_deg = TOOTH_ANGLES_DEG,
    # Tooth 31 (3-slot gap, ~26.5°) following a normal tooth (~10°): ratio ≈ 2.65
    # Wide window to accommodate measurement noise and variable speed.
    # Mirrors setTriggerSynchronizationGap() / synchronizationRatioFrom[0] in trigger_mitsubishi.cpp
    sync_ratio_from  = [1.8, 0.5],
    sync_ratio_to    = [3.6, 1.5],
	#sync_ratio_from  = [1.8],
    #sync_ratio_to    = [3.6],
    sync_gap_deg     = float(TOOTH_ANGLES_DEG[-1]),  # 26.5°
)

# Choose an RPM profile - uncomment the one you want:
# syn_rpm_profile = rpm_const(SYN_RPM)
# syn_rpm_profile = rpm_linear(SYN_RPM, accel_rpm_per_rev=20)          # constant acceleration: +20 RPM/rev
# syn_rpm_profile = rpm_sine(SYN_RPM, amplitude=90, period_deg=120, phase_deg=120)
RAMP_UP_MS = 300.0
RAMP_UP_ACCEL_RPM_PER_MS = 20.0
RAMP_HOLD_MS = 700.0
RAMP_DOWN_MS = 300.0
RAMP_DOWN_ACCEL_RPM_PER_MS = -30.0

ramp_up_end_rpm = SYN_RPM + RAMP_UP_ACCEL_RPM_PER_MS * RAMP_UP_MS
ramp_down_end_rpm = max(ramp_up_end_rpm + RAMP_DOWN_ACCEL_RPM_PER_MS * RAMP_DOWN_MS, 1.0)

syn_rpm_profile = rpm_staged_ms([
    (RAMP_UP_MS, rpm_linear_ms(SYN_RPM, accel_rpm_per_ms=RAMP_UP_ACCEL_RPM_PER_MS)),
    (RAMP_HOLD_MS, rpm_const(ramp_up_end_rpm)),
    (RAMP_DOWN_MS, rpm_linear_ms(ramp_up_end_rpm, accel_rpm_per_ms=RAMP_DOWN_ACCEL_RPM_PER_MS)),
    (None, rpm_const(ramp_down_end_rpm)),
])
# syn_rpm_profile = rpm_sine(SYN_RPM, amplitude=30, period_deg=180)    # +-30 RPM sine, 2 cycles/rev

# ── Generate waveform & run filter ────────────────────────────────────────────
syn_times, syn_intervals = gen_synthetic_waveform(syn_rpm_profile, SYN_N_REVS, WHEEL_6G75)
syn_signal_times, syn_signal_values = gen_synthetic_signal(syn_times, syn_intervals)

_n = len(WHEEL_6G75.tooth_angles_deg)
_angles = WHEEL_6G75.tooth_angles_deg[np.arange(len(syn_intervals)) % _n]
rpm_from_intervals = 60_000.0 / (360.0 * syn_intervals / _angles)
print(f"Synthetic signal: {SYN_N_REVS} revolutions, {len(syn_times)} edges,  duration = {syn_times.iloc[-1]:.1f} ms")
print(f"RPM range: {rpm_from_intervals.min():.1f} - {rpm_from_intervals.max():.1f}")
print(f"Ramp-up end RPM: {ramp_up_end_rpm:.1f}; Ramp-down end RPM: {ramp_down_end_rpm:.1f}")


Tooth angles [deg]: [13.5 10.  10.  10.  10.  10.  10.  10.  10.  10.  16.5 13.5 10.  10.
 10.  10.  10.  10.  10.  10.  10.  16.5 13.5 10.  10.  10.  10.  10.
 10.  10.  10.  26.5]
Sum = 360.0°
Synthetic signal: 100 revolutions, 3200 edges,  duration = 1144.7 ms
RPM range: 300.0 - 6300.0
Ramp-up end RPM: 6300.0; Ramp-down end RPM: 1.0


In [345]:
files = [
	'6g75-without-spark-crank.csv',
	'6g75-withsparkplugs-cranking.csv',
]

csv_path = r"../../unit_tests/tests/trigger/resources/" + files[1]
df = pd.read_csv(csv_path)

df.head()

n_from = 2
n_to = 2400

time = df.iloc[n_from:n_to, 0]
ch0 = df.iloc[n_from:n_to, 1]

signal = df.iloc[n_from:n_to, 1].reset_index(drop=True)
time_arr = df.iloc[n_from:n_to, 0].reset_index(drop=True) * 1000  # convert to ms

# or use synthetic signal:
#signal = syn_signal_values
#time_arr = syn_signal_times

# Optional: inject random glitch pulses into the waveform.
# Operates on signal/time_arr directly so all downstream code (edge detector,
# decoder, plots) sees the glitches without any changes.
#
#time_arr, signal = inject_signal_glitches(time_arr, signal, n_glitches=20, glitch_width=1, seed=42)  # 1 ms wide glitches
#time_arr, signal = drop_signal_teeth(time_arr, signal, n_drops=5, seed=42)

time_ms = time_arr  # already in milliseconds

In [346]:

fig = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.08, subplot_titles=("Channel 0", "Channel 1"))

fig.add_trace(go.Scatter(x=time_ms, y=signal, mode='lines', line=dict(width=1, shape='hv'), name='Channel 0'), row=1, col=1)

fig.update_layout(title="6G75 Trigger Signals", height=500, hovermode='x unified')
fig.update_xaxes(title_text="Time [ms]", row=1, col=1)

fig.show()


In [347]:
# Fast rising-edge detection and bounded plotting
EDGE_THRESHOLD = 0.5
MAX_PLOT_EDGES = 100  # Set to None to plot all detected edges

# Use NumPy arrays for faster edge detection than pandas diff/masking
signal_arr = np.asarray(signal, dtype=float)
time_ms_arr = np.asarray(time_ms, dtype=float)

rising_idx = np.flatnonzero((signal_arr[1:] > EDGE_THRESHOLD) & (signal_arr[:-1] <= EDGE_THRESHOLD)) + 1
rising_times_arr = time_ms_arr[rising_idx]
rising_times_ms = pd.Series(rising_times_arr)

total_edges = len(rising_times_arr)
if MAX_PLOT_EDGES is None or MAX_PLOT_EDGES <= 0 or MAX_PLOT_EDGES >= total_edges:
    plot_edges = total_edges
else:
    plot_edges = int(MAX_PLOT_EDGES)

print(f"Found {total_edges} rising edges")
if plot_edges < total_edges:
    print(f"Plotting first {plot_edges} rising edges (set MAX_PLOT_EDGES=None to plot all)")

plot_rising_idx = rising_idx[:plot_edges]
plot_rising_times = time_ms_arr[plot_rising_idx]

# Limit the actual plotted waveform data to the same edge window (+1 edge interval margin).
if plot_edges > 0:
    first_plot_idx = int(plot_rising_idx[0])
    if plot_edges < total_edges:
        next_after_last = int(rising_idx[plot_edges])
        last_plot_idx = min(len(signal_arr), next_after_last + 1)
    else:
        last_plot_idx = len(signal_arr)
else:
    first_plot_idx = 0
    last_plot_idx = len(signal_arr)

plot_time_arr = time_ms_arr[first_plot_idx:last_plot_idx]
plot_signal_arr = signal_arr[first_plot_idx:last_plot_idx]

fig2 = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=("Signal with rising edges", "Period between rising edges", "Period ratio (current / previous)", "Ratio of ratios (2nd derivative)")
)

fig2.add_trace(
    go.Scatter(
        x=plot_time_arr, y=plot_signal_arr,
        mode='lines', line=dict(width=1, shape='hv'), name='Channel 0'
    ),
    row=1, col=1
)

# Use one vline shape per displayed edge only (major plotting cost when unbounded)
for t in plot_rising_times:
    fig2.add_shape(
        type='line', x0=t, x1=t, y0=0, y1=1, xref='x', yref='paper',
        opacity=0.3, line=dict(color='red', width=1, dash='dash')
    )

fig2.add_trace(
    go.Scatter(
        x=plot_rising_times,
        y=np.full(plot_edges, float(np.max(plot_signal_arr))) if plot_edges > 0 else [],
        mode='markers', marker=dict(color='red', symbol='triangle-down', size=10),
        name=f'Rising edges shown ({plot_edges}/{total_edges})'
    ),
    row=1, col=1
)

# Compute period features from displayed edges to keep plotting bounded
periods_ms = pd.Series(np.diff(plot_rising_times))
period_times_ms = pd.Series(plot_rising_times[1:])
ratios = periods_ms / periods_ms.shift(1)
ratio_of_ratios = ratios / ratios.shift(1)

fig2.add_trace(go.Scatter(
    x=period_times_ms, y=periods_ms,
    mode='lines+markers', marker=dict(size=4), line=dict(width=1, shape='hv'),
    name='Period [ms]'
), row=2, col=1)

fig2.add_trace(go.Scatter(
    x=period_times_ms, y=ratios,
    mode='lines+markers', marker=dict(size=4), line=dict(width=1, shape='hv'),
    name='Period ratio'
), row=3, col=1)

fig2.add_trace(go.Scatter(
    x=period_times_ms, y=ratio_of_ratios,
    mode='lines+markers', marker=dict(size=4), line=dict(width=1, shape='hv'),
    name='Ratio of ratios'
), row=4, col=1)

fig2.update_yaxes(title_text="Signal", row=1, col=1)
fig2.update_yaxes(title_text="Period [ms]", row=2, col=1)
fig2.update_yaxes(title_text="Ratio", row=3, col=1)
fig2.update_yaxes(title_text="Ratio^2", row=4, col=1)
fig2.update_xaxes(title_text="Time [ms]", row=4, col=1)
fig2.update_layout(title="6G75 Trigger Signal - Rising Edges, Periods & Ratios", height=1000, hovermode='x unified')
fig2.show()


Found 420 rising edges
Plotting first 100 rising edges (set MAX_PLOT_EDGES=None to plot all)


In [359]:

# ── rusEFI-style trigger decoder prototype ─────────────────────────────────────
#
# Mirrors TriggerDecoderBase / decodeTriggerEvent() as closely as possible:
#   * Stream processing: on_edge(dt_ms) called once per rising edge
#   * No numpy in the hot path — pure scalar arithmetic
#   * toothDurations[] ring buffer — shift-register, index 0 = most recent
#   * isSyncPoint() — checks ratio windows from WheelDescriptor (wheel-independent)
#   * Post-sync: alpha-beta tracker with per-tooth angle model
#
# Sync lifecycle:
#   PRE-SYNC    shaft_is_synchronized=False — waiting for isSyncPoint()
#   CANDIDATE   shaft_is_synchronized=True, tracker_ok=False (warmup period)
#               gate uses warmup_gate_k (wider); exceedance → reset
#   CONFIRMED   shaft_is_synchronized=True, tracker_ok=True
#               gate-exceeding innovations → glitch absorbed; 2 in a row → reset
#
# Warmup criterion: accumulated crank angle ≥ warmup_angle_deg (not tooth count).
# Angle-based warmup is robust to sparse/non-uniform wheels: a wheel with a large
# sync-gap tooth (e.g. 26.5°) would count the same as a tiny 10° tooth under a
# tooth-count scheme — angle accumulation treats them fairly.
#
# Glitch absorption (confirmed mode):
#   A single detected glitch: its interval (dt_eff) is carried forward via
#   pending_dt so the NEXT real tooth sees a full-period measurement.
#   tooth_idx is NOT advanced — the spurious edge does not consume a tooth slot.
#   Two consecutive glitches: sync is considered lost and is reset immediately.
#
# Per-revolution phase validation:
#   When tooth_idx reaches n_teeth-1 (the expected sync-gap position), _is_sync_point()
#   is called to confirm the ratio pattern still matches.  A missing tooth shifts
#   phase by one slot without generating a large innovation, so this check catches
#   phase drift that the glitch gate alone would miss.  Reset is immediate on failure.
#
# Python ↔ C++ field mapping:
#   tooth_durations[i]          ≡  toothDurations[i]          (TriggerDecoderBase)
#   shaft_is_synchronized       ≡  shaft_is_synchronized       (TriggerDecoderBase)
#   wheel.sync_ratio_from/to[i] ≡  synchronizationRatioFrom/To[i] (TriggerWaveform)
#   wheel.sync_gap_deg          ≡  angle of sync-gap tooth used for tracker bootstrap
#   t_hat / t_dot_hat           ≡  rate/trend EMA state (proposed new fields)
#   sync_var                    ≡  innovation-variance EMA (proposed new field)

def make_decoder(wheel,
                 alpha_rate       = 0.125,   # rate EMA gain   (1/8  → right-shift 3)
                 alpha_trend      = 0.0625,  # trend EMA gain  (1/16 → right-shift 4)
                 alpha_var        = 0.0625,  # variance EMA gain
                 gate_k           = 9,       # glitch gate multiplier in confirmed mode (≈ 3σ²)
                 warmup_gate_k    = None,    # gate multiplier during candidate warmup;
                                             # None → 4 × gate_k (wider, more forgiving)
                 var_floor_frac   = 0.05,    # sync_var floor = (var_floor_frac * t_hat)²
                 warmup_angle_deg = 360.0):  # accumulated crank angle [deg] before CANDIDATE → CONFIRMED
    """
    Returns  on_edge(dt_ms) -> dict  — call once per rising edge.

    Wheel-independent: sync detection reads synchronizationRatioFrom/To
    from the WheelDescriptor, not from decoder parameters.

    The candidate (warmup) window lasts until the accumulated crank angle
    since sync acquisition reaches warmup_angle_deg.  Using angle rather than
    tooth count means sparse-tooth wheels (large sync-gap tooth) get the same
    angular coverage as dense-tooth wheels.

    Output dict fields:
        shaft_is_synchronized  — True once sync is acquired and stable
        is_sync                — True on the edge that triggers sync acquisition
        is_glitch              — True if confirmed-sync glitch gate fired
        is_sync_reset          — True if sync was dropped (candidate gate, 2× glitch, or phase mismatch)
        is_candidate           — True while in warmup (candidate sync window)
        tooth_idx, innov, r_hat, t_dot, sync_var, rpm, warmup_angle_acc
    """
    if warmup_gate_k is None:
        warmup_gate_k = gate_k * 4

    n_teeth          = len(wheel.tooth_angles_deg)
    gap_tracking_len = len(wheel.sync_ratio_from)   # ≡ gapTrackingLength in TriggerWaveform

    # toothDurations[GAP_TRACKING_LENGTH + 1] — [0]=current, [1]=previous, …
    tooth_durations = [0.0] * (gap_tracking_len + 1)

    shaft_is_synchronized = False
    tooth_idx            = 0
    t_hat                = 0.0
    t_dot_hat            = 0.0
    sync_var             = 0.0
    tracker_ok           = False
    warmup_angle_acc     = 0.0   # accumulated crank angle since sync acquisition [deg]
    pending_dt           = 0.0   # time carried forward from a glitch edge
    consecutive_glitches = 0     # resets sync when ≥ 2 in a row

    def _reset_sync():
        nonlocal shaft_is_synchronized, tooth_idx
        nonlocal t_hat, t_dot_hat, sync_var, tracker_ok, warmup_angle_acc
        nonlocal pending_dt, consecutive_glitches
        shaft_is_synchronized = False
        tooth_idx            = 0
        t_hat                = 0.0
        t_dot_hat            = 0.0
        sync_var             = 0.0
        tracker_ok           = False
        warmup_angle_acc     = 0.0
        pending_dt           = 0.0
        consecutive_glitches = 0

    def _is_sync_point():
        """
        Mirrors TriggerDecoderBase::isSyncPoint().
        Checks that tooth_durations[i] / tooth_durations[i+1] is within
        [sync_ratio_from[i], sync_ratio_to[i]] for every tracked gap.
        """
        for i in range(gap_tracking_len):
            if tooth_durations[i + 1] == 0.0:
                return False
            ratio = tooth_durations[i] / tooth_durations[i + 1]
            if not (wheel.sync_ratio_from[i] <= ratio <= wheel.sync_ratio_to[i]):
                return False
        return True

    def on_edge(dt):
        nonlocal shaft_is_synchronized, tooth_idx
        nonlocal t_hat, t_dot_hat, sync_var, tracker_ok, warmup_angle_acc
        nonlocal pending_dt, consecutive_glitches

        # Absorb any time carried forward from a previous glitch edge.
        dt_eff     = dt + pending_dt
        pending_dt = 0.0

        # Shift tooth_durations ring buffer
        for i in range(gap_tracking_len, 0, -1):
            tooth_durations[i] = tooth_durations[i - 1]
        tooth_durations[0] = dt_eff

        innov         = 0.0
        is_sync       = False
        is_glitch     = False
        is_sync_reset = False

        if not shaft_is_synchronized:
            # ── PRE-SYNC ──────────────────────────────────────────────────────
            if _is_sync_point():
                shaft_is_synchronized = True
                tooth_idx        = 0
                is_sync          = True
                t_hat            = dt_eff / wheel.sync_gap_deg
                t_dot_hat        = 0.0
                sync_var         = 0.0
                warmup_angle_acc = 0.0
                tracker_ok       = False

            r_hat_out = t_hat

        else:
            # ── POST-SYNC: alpha-beta tracker ─────────────────────────────────

            # Step 1 — Predict
            angle_deg = wheel.tooth_angles_deg[tooth_idx]
            r_pred = t_hat
            r_meas = dt_eff / angle_deg
            innov  = r_meas - r_pred

            # Step 2 — Glitch gate
            # Use warmup_gate_k during candidate mode (wider, more forgiving),
            # gate_k once the tracker is confirmed (tighter).
            active_gate_k = warmup_gate_k if not tracker_ok else gate_k
            exceeds_gate = sync_var > 0.0 and (innov * innov) > active_gate_k * sync_var
            if exceeds_gate:
                if not tracker_ok:
                    # CANDIDATE: gate fire → reset immediately
                    _reset_sync()
                    return {
                        'shaft_is_synchronized': False,
                        'is_sync':         False,
                        'is_glitch':       False,
                        'is_sync_reset':   True,
                        'is_candidate':    False,
                        'tooth_idx':       0,
                        'innov':           innov,
                        'r_hat':           0.0,
                        't_dot':           0.0,
                        'sync_var':        0.0,
                        'rpm':             0.0,
                        'warmup_angle_acc': 0.0,
                    }
                else:
                    is_glitch = True

            if is_glitch:
                # CONFIRMED: absorb a single glitch; two in a row → reset sync
                consecutive_glitches += 1
                if consecutive_glitches >= 2:
                    _reset_sync()
                    return {
                        'shaft_is_synchronized': False,
                        'is_sync':         False,
                        'is_glitch':       True,
                        'is_sync_reset':   True,
                        'is_candidate':    False,
                        'tooth_idx':       0,
                        'innov':           innov,
                        'r_hat':           0.0,
                        't_dot':           0.0,
                        'sync_var':        0.0,
                        'rpm':             0.0,
                        'warmup_angle_acc': 0.0,
                    }
                # Single glitch: carry interval forward, don't advance tooth_idx
                pending_dt = dt_eff
                return {
                    'shaft_is_synchronized': shaft_is_synchronized,
                    'is_sync':         False,
                    'is_glitch':       True,
                    'is_sync_reset':   False,
                    'is_candidate':    not tracker_ok,
                    'tooth_idx':       tooth_idx,
                    'innov':           innov,
                    'r_hat':           t_hat,
                    't_dot':           t_dot_hat,
                    'sync_var':        sync_var,
                    'rpm':             60_000.0 / (360.0 * t_hat) if t_hat > 0.0 else 0.0,
                    'warmup_angle_acc': warmup_angle_acc,
                }

            # Step 3 — Update (normal tooth)
            consecutive_glitches = 0   # clean tooth resets the consecutive-glitch streak
            t_hat     += alpha_rate  * innov
            t_dot_hat += alpha_trend * innov
            sync_var  += alpha_var   * (innov * innov - sync_var)

            # Floor applied AFTER EMA so the EMA cannot drive sync_var below it.
            # Gate width ≥ var_floor_frac × t_hat ensures real tooth tolerances
            # are never mistaken for glitches.
            floor_val = (var_floor_frac * t_hat) ** 2
            if sync_var < floor_val:
                sync_var = floor_val

            # Accumulate crank angle for warmup criterion (angle-based, not tooth count)
            warmup_angle_acc += angle_deg
            if warmup_angle_acc >= warmup_angle_deg:
                tracker_ok = True

            r_hat_out = t_hat   # log AFTER update, BEFORE predict

            # Predict-step: advance t_hat by trend for next tooth
            t_hat    += t_dot_hat
            tooth_idx = (tooth_idx + 1) % n_teeth

        rpm = 60_000.0 / (360.0 * t_hat) if t_hat > 0.0 else 0.0
        return {
            'shaft_is_synchronized': shaft_is_synchronized,
            'is_sync':         is_sync,
            'is_glitch':       is_glitch,
            'is_sync_reset':   is_sync_reset,
            'is_candidate':    shaft_is_synchronized and not tracker_ok,
            'tooth_idx':       tooth_idx,
            'innov':           innov,
            'r_hat':           r_hat_out,
            't_dot':           t_dot_hat,
            'sync_var':        sync_var,
            'rpm':             rpm,
            'warmup_angle_acc': warmup_angle_acc,
        }

    return on_edge


print("Defined: make_decoder(wheel)")
print("Usage:  decoder = make_decoder(WHEEL_6G75)")
print("        entry   = decoder(dt_ms)   # once per rising edge")


Defined: make_decoder(wheel)
Usage:  decoder = make_decoder(WHEEL_6G75)
        entry   = decoder(dt_ms)   # once per rising edge


In [362]:

# ── Run prototype decoder and plot ────────────────────────────────────────────

PROTO_ALPHA_RATE      = 0.125
PROTO_ALPHA_TREND     = 0.125
PROTO_ALPHA_VAR       = 0.0625
PROTO_GATE_K          = 9
PROTO_WARMUP_GATE_K   = None    # None → 4 × PROTO_GATE_K (wider gate during warmup)
PROTO_VAR_FLOOR_FRAC  = 0.05   # gate never narrows below 5% of r̂  (≈ real-world tooth tolerance floor)
PROTO_WARMUP_ANGLE_DEG = 360.0  # crank degrees to accumulate before CANDIDATE → CONFIRMED

decoder = make_decoder(
    WHEEL_6G75,
    alpha_rate       = PROTO_ALPHA_RATE,
    alpha_trend      = PROTO_ALPHA_TREND,
    alpha_var        = PROTO_ALPHA_VAR,
    gate_k           = PROTO_GATE_K,
    warmup_gate_k    = PROTO_WARMUP_GATE_K,
    var_floor_frac   = PROTO_VAR_FLOOR_FRAC,
    warmup_angle_deg = PROTO_WARMUP_ANGLE_DEG,
)

# Stream: one inter-edge interval at a time, no bulk numpy operations
times_arr = rising_times_ms.values
log = []
for k in range(1, len(times_arr)):
    dt    = times_arr[k] - times_arr[k - 1]
    entry = decoder(dt)
    entry['t'] = times_arr[k]
    log.append(entry)

# ── Collect output into plain lists for plotting ───────────────────────────────
t_out              = [e['t']                     for e in log]
rpm_out            = [e['rpm']                   for e in log]
innov_out          = [e['innov']                 for e in log]
tdot_out           = [e['t_dot']                 for e in log]
synced_out         = [e['shaft_is_synchronized'] for e in log]
glitch_out         = [e['is_glitch']             for e in log]
sync_var_out       = [e['sync_var']              for e in log]
candidate_out      = [e['is_candidate']          for e in log]
reset_out          = [e['is_sync_reset']         for e in log]
warmup_angle_out   = [e['warmup_angle_acc']      for e in log]

import math
resolved_warmup_gate_k = PROTO_WARMUP_GATE_K if PROTO_WARMUP_GATE_K is not None else PROTO_GATE_K * 4
gate_thresh         = [math.sqrt(PROTO_GATE_K         * sv) if sv > 0 else 0.0 for sv in sync_var_out]
warmup_gate_thresh  = [math.sqrt(resolved_warmup_gate_k * sv) if sv > 0 else 0.0 for sv in sync_var_out]

print(f"Edges after confirmed sync: {sum(synced_out)}")
print(f"Glitches flagged (confirmed mode): {sum(glitch_out)}")
print(f"Sync resets: {sum(reset_out)}")
print(f"Warmup gate_k: {resolved_warmup_gate_k}  (confirmed gate_k: {PROTO_GATE_K})")
print(f"Warmup angle: {PROTO_WARMUP_ANGLE_DEG}°")

# ── Candidate windows: spans where is_candidate was True ──────────────────────
candidate_spans = []
cand_start = None
for t, cand in zip(t_out, candidate_out):
    if cand and cand_start is None:
        cand_start = t
    elif not cand and cand_start is not None:
        candidate_spans.append((cand_start, t))
        cand_start = None
if cand_start is not None:
    candidate_spans.append((cand_start, t_out[-1]))

# ── Unsynced windows: spans where shaft_is_synchronized was False ──────────────
unsync_spans = []
unsync_start = t_out[0] if not synced_out[0] else None
for i in range(1, len(t_out)):
    was_synced = synced_out[i - 1]
    is_synced  = synced_out[i]
    if was_synced and not is_synced:
        unsync_start = t_out[i]
    elif not was_synced and is_synced and unsync_start is not None:
        unsync_spans.append((unsync_start, t_out[i]))
        unsync_start = None
if unsync_start is not None:
    unsync_spans.append((unsync_start, t_out[-1]))

# ── Plot ───────────────────────────────────────────────────────────────────────
fig_proto = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.07,
    subplot_titles=(
        "Signal",
        "Innovation [ms/deg]",
        "RPM",
        "Warmup angle accumulated [deg]",
    )
)

# Row 1: signal
fig_proto.add_trace(
    go.Scatter(x=time_ms, y=signal, mode='lines',
               line=dict(width=1, shape='hv'), name='Signal'),
    row=1, col=1
)

# Row 2: innovations + confirmed gate band
fig_proto.add_trace(
    go.Scatter(x=t_out, y=gate_thresh, mode='lines',
               line=dict(width=0), showlegend=False, hoverinfo='skip'),
    row=2, col=1
)
fig_proto.add_trace(
    go.Scatter(x=t_out, y=[-v for v in gate_thresh], mode='lines',
               line=dict(width=0), fill='tonexty',
               fillcolor='rgba(255,165,0,0.25)',
               name='±√(gate_k·var)'),
    row=2, col=1
)

# Warmup gate band (wider, dotted blue lines)
fig_proto.add_trace(
    go.Scatter(x=t_out, y=warmup_gate_thresh, mode='lines',
               line=dict(width=1, color='rgba(100,100,255,0.5)', dash='dot'),
               name='±√(warmup_gate_k·var)', showlegend=True),
    row=2, col=1
)
fig_proto.add_trace(
    go.Scatter(x=t_out, y=[-v for v in warmup_gate_thresh], mode='lines',
               line=dict(width=1, color='rgba(100,100,255,0.5)', dash='dot'),
               showlegend=False, hoverinfo='skip'),
    row=2, col=1
)

fig_proto.add_trace(
    go.Scatter(x=t_out, y=innov_out, mode='lines+markers',
               marker=dict(size=3), line=dict(width=1, shape='hv'),
               name='Innovation'),
    row=2, col=1
)
fig_proto.add_hline(y=0, line=dict(color='gray', width=1, dash='dot'), row=2, col=1)

# Glitches (confirmed mode)
glitch_t   = [t_out[i] for i, g in enumerate(glitch_out) if g]
glitch_inn = [innov_out[i] for i, g in enumerate(glitch_out) if g]
if glitch_t:
    fig_proto.add_trace(
        go.Scatter(x=glitch_t, y=glitch_inn, mode='markers',
                   marker=dict(color='red', size=8, symbol='x'), name='Glitch'),
        row=2, col=1
    )

# Sync resets: orange X on innovation row
reset_t     = [t_out[i] for i, r in enumerate(reset_out) if r]
reset_innov = [innov_out[i] for i, r in enumerate(reset_out) if r]
if reset_t:
    fig_proto.add_trace(
        go.Scatter(x=reset_t, y=reset_innov, mode='markers',
                   marker=dict(color='orange', size=10, symbol='x'),
                   name='Sync reset'),
        row=2, col=1
    )

# Row 3: RPM
fig_proto.add_trace(
    go.Scatter(x=t_out, y=rpm_out, mode='lines+markers',
               marker=dict(size=3), line=dict(width=1, shape='hv'),
               name='RPM'),
    row=3, col=1
)

# Row 4: warmup angle accumulator (ramps up during candidate, holds at threshold after confirmation)
fig_proto.add_trace(
    go.Scatter(x=t_out, y=warmup_angle_out, mode='lines+markers',
               marker=dict(size=3), line=dict(width=1, shape='hv'),
               name='Warmup angle acc [deg]'),
    row=4, col=1
)
fig_proto.add_hline(y=PROTO_WARMUP_ANGLE_DEG,
                    line=dict(color='green', width=1, dash='dash'),
                    annotation_text=f'{PROTO_WARMUP_ANGLE_DEG}° threshold',
                    row=4, col=1)

# Unsynced windows: translucent red shading (drawn first so green sits on top)
for us, ue in unsync_spans:
    fig_proto.add_vrect(
        x0=us, x1=ue,
        fillcolor='rgba(220,50,50,0.12)', layer='below', line_width=0,
    )

# Candidate windows: light green shading
for cs, ce in candidate_spans:
    fig_proto.add_vrect(
        x0=cs, x1=ce,
        fillcolor='rgba(0,200,80,0.15)', layer='below', line_width=0,
    )

# Sync acquisition markers (green dashed lines)
for e in log:
    if e['is_sync']:
        fig_proto.add_vline(x=e['t'], line=dict(color='green', width=2, dash='dash'),
                            annotation_text='sync', row='all', col='all')

fig_proto.update_yaxes(title_text="Signal",         row=1, col=1)
fig_proto.update_yaxes(title_text="Innov [ms/deg]", row=2, col=1)
fig_proto.update_yaxes(title_text="RPM",            row=3, col=1)
fig_proto.update_yaxes(title_text="Angle [deg]",    row=4, col=1)
fig_proto.update_xaxes(title_text="Time [ms]",      row=4, col=1)
fig_proto.update_layout(
    title=(
        f"α_rate={PROTO_ALPHA_RATE}, α_trend={PROTO_ALPHA_TREND}, "
        f"var_floor={PROTO_VAR_FLOOR_FRAC*100:.0f}%, "
        f"gate_k={PROTO_GATE_K}, warmup_gate_k={resolved_warmup_gate_k}, "
        f"warmup={PROTO_WARMUP_ANGLE_DEG}°"
    ),
    height=1000, hovermode='x unified'
)
fig_proto.show()


Edges after confirmed sync: 407
Glitches flagged (confirmed mode): 2
Sync resets: 2
Warmup gate_k: 36  (confirmed gate_k: 9)
Warmup angle: 360.0°
